In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [26]:

from llama_index.core import Document
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core import load_index_from_storage
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.indices.postprocessor import SentenceTransformerRerank
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.embeddings.google_genai.base import types

documents = SimpleDirectoryReader(input_files = ["data/constitution of india.pdf"]).load_data()
document = [Document(text="\n\n".join([doc.text for doc in documents]))]

def get_auto_merging_index(document, index_dir, chunk_sizes=[2048, 512, 128]):
    
    node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes = chunk_sizes)
    nodes = node_parser.get_nodes_from_documents(document)
    leaf_nodes = get_leaf_nodes(nodes)

    Settings.llm = GoogleGenAI(
    model="gemini-2.5-flash",
    generation_config=types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(
                category= types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_CIVIC_INTEGRITY,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                threshold=types.HarmBlockThreshold.BLOCK_NONE
            ),
        ]
    ))
    Settings.embed_model = GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        embedding_config=types.EmbedContentConfig(
            output_dimensionality=1536,
            task_type="RETRIEVAL_DOCUMENT"
        )
    )
    Settings.node_parser = node_parser
    
    docstore = SimpleDocumentStore()

    # insert nodes into docstore
    docstore.add_documents(nodes)

    # define storage context (will include vector store by default too)
    storage_context = StorageContext.from_defaults(docstore=docstore)
    
    if not os.path.exists(index_dir):
        automerging_index = VectorStoreIndex(leaf_nodes, storage_context=storage_context)
        automerging_index.storage_context.persist(persist_dir=index_dir)
    else:
        automerging_index = load_index_from_storage(StorageContext.from_defaults(persist_dir=index_dir))

    return automerging_index

def get_auto_merging_engine(am_index):
    
    base_retriever = am_index.as_retriever(similarity_top_k=6)
    retriever = AutoMergingRetriever(base_retriever, am_index.storage_context, verbose=True)
  
    auto_merging_engine = RetrieverQueryEngine.from_args(retriever)
    
    return auto_merging_engine

In [27]:
index_dir = "automerging_index_3"
am_index_3 = get_auto_merging_index(document, index_dir, chunk_sizes=[2048, 512, 128])
am_engine_3 = get_auto_merging_engine(am_index_3)

2025-09-21 20:44:44,342 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash "HTTP/1.1 200 OK"
2025-09-21 20:44:47,368 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-21 20:44:48,565 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-21 20:44:49,805 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-21 20:44:51,191 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-21 20:44:52,461 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-21 20:44:53,752 - IN

In [29]:
window_response_2 = am_engine_3.query(
    "Explain the scope of Article 21 of the Indian Constitution"
)
window_response_2.response

2025-09-21 20:56:35,930 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2025-09-21 20:56:36,100 - INFO - > Merging 3 nodes into parent node.
> Parent node id: 933df88f-069e-4291-9d29-71fdbd7d0242.
> Parent node text: 20. Protection in respect of conviction for offences. —(1) No person 
shall be convicted of any o...

2025-09-21 20:56:36,105 - INFO - AFC is enabled with max remote calls: 10.


> Merging 3 nodes into parent node.
> Parent node id: 933df88f-069e-4291-9d29-71fdbd7d0242.
> Parent node text: 20. Protection in respect of conviction for offences. —(1) No person 
shall be convicted of any o...



2025-09-21 20:56:39,172 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2025-09-21 20:56:39,174 - INFO - AFC remote call 1 is done.


'Article 21 ensures the protection of life and personal liberty. It states that no individual can be deprived of their life or personal liberty except according to a procedure established by law.'